# watsonx.data Spark Integration - Lab Notebook

This notebook demonstrates how to connect to watsonx.data from a local Spark environment and perform basic operations with Iceberg tables.

**Prerequisites:**
- watsonx.data instance with Hive Metastore (HMS)
- Valid watsonx.data username and API key
- PySpark installed locally
- Network access to watsonx.data HMS

## 1. Import Required Libraries

Import all necessary Python libraries for Spark, authentication, and utilities.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_utc_timestamp
import os
import base64
import getpass
import re

## 2. Collect Credentials

Securely prompt for watsonx.data credentials. The API key is used for both HMS authentication and Spark extension authentication.

In [ ]:
# Prompt for watsonx.data username
wxd_username = getpass.getpass("Please enter your watsonx.data username (HMS access): ").strip()

# HMS expects username in this format for API key authentication
wxd_hms_username = f"ibmlhapikey_{wxd_username}"

# Prompt for API key (used for metastore access)
wxd_hms_password = getpass.getpass("Please enter your watsonx.data API key (HMS access): ").strip()

# Encode API key in ZenApiKey format for Spark extension
string_to_encode = f"{wxd_username}:{wxd_hms_password}"
wxd_encoded_apikey = "ZenApiKey " + base64.b64encode(string_to_encode.encode("utf-8")).decode("utf-8")

print("✓ Credentials collected successfully")

## 3. Initialize Spark Session

Create a Spark session configured for watsonx.data with:
- HMS authentication
- Iceberg Spark extensions
- Hive support enabled

In [ ]:
def init_spark():
    """
    Initialize Spark session with watsonx.data configuration.
    
    Returns:
        SparkSession: Configured Spark session
    """
    spark = SparkSession.builder \
        .appName("vscode-demo") \
        .config("spark.hive.metastore.client.plain.username", wxd_hms_username) \
        .config("spark.hive.metastore.client.plain.password", wxd_hms_password) \
        .config("spark.hadoop.hive.wxd.user.name", wxd_username) \
        .config("spark.hadoop.wxd.apikey", wxd_encoded_apikey) \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.timestampType", "TIMESTAMP_NTZ") \
        .enableHiveSupport() \
        .getOrCreate()
    
    return spark

# Initialize Spark session
spark = init_spark()

print(f"✓ Spark session initialized")
print(f"  Spark version: {spark.version}")

## 4. Verify Metastore Connectivity

Test connection to watsonx.data Hive Metastore by listing available databases.

In [ ]:
try:
    # List all databases in the metastore
    dbs = spark.catalog.listDatabases()
    
    print(f"✅ Metastore reachable. Databases found: {len(dbs)}")
    print("\nAvailable databases:")
    
    for d in dbs[:50]:  # Limit to first 50 databases
        print(f"  - {d.name}")
        
except Exception as e:
    print("❌ Metastore NOT reachable")
    print(f"Exception type: {type(e).__name__}")
    print(f"Error: {str(e)}")
    
    # Print additional error details if available
    if hasattr(e, "desc"):
        print("\n--- Error Description ---")
        print(e.desc)
    
    if hasattr(e, "stackTrace"):
        print("\n--- Stack Trace (first 2500 chars) ---")
        print(str(e.stackTrace)[:2500])
    
    raise

## 5. Discover Available Catalogs

Extract catalog names from Spark configuration to identify available data sources.

In [ ]:
# Get all Spark configuration items
conf_items = dict(spark.sparkContext.getConf().getAll())

# Pattern to match catalog configuration keys
catalog_pattern = re.compile(r"^spark\.sql\.catalog\.([^.]+)(?:\..+)?$")

# Extract unique catalog names
catalogs = sorted(
    {m.group(1) for k in conf_items.keys() if (m := catalog_pattern.match(k))} | {"spark_catalog"}
)

print("Catalogs discovered from Spark config:")
for c in catalogs:
    print(f"  - {c}")

# Also show catalogs via SQL
print("\nCatalogs via SHOW CATALOGS:")
spark.sql("SHOW CATALOGS").show(truncate=False)

## 6. List Databases in Iceberg Catalog

Query the iceberg_data catalog to see available schemas/databases.

In [ ]:
def list_databases(spark, catalog="iceberg_data"):
    """
    List all databases in the specified catalog.
    
    Args:
        spark: SparkSession instance
        catalog: Catalog name (default: iceberg_data)
    """
    print(f"Databases in '{catalog}' catalog:")
    spark.sql(f"SHOW DATABASES IN {catalog}").show(truncate=False)

list_databases(spark)

## 7. Create Schema/Database

Create a new schema in the iceberg_data catalog with a specific S3 location.

In [ ]:
# Define schema details
catalog_name = "iceberg_data"  # Note: underscore, not hyphen
schema_name = "hednolab01"
schema_location = f"s3a://iceberg-bucket/{schema_name}"

# Create schema if it doesn't exist
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
    LOCATION '{schema_location}'
""")

print(f"✓ Schema ensured: {catalog_name}.{schema_name}")
print(f"  Location: {schema_location}")

## 8. Create Iceberg Table - Simple Example

Create a basic Iceberg table with standard columns.

**Important:** If catalog name contains hyphens, use backticks for quoting.

In [ ]:
# Define table details
table_name = "demo_table_10rows"

# Use backticks if catalog name has hyphens (e.g., `iceberg-data`)
# Use regular name if catalog has underscores (e.g., iceberg_data)
full_table = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Creating table: {full_table}")

# Create table with basic schema
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {full_table} (
        id INT COMMENT 'Unique identifier',
        name STRING COMMENT 'Record name',
        amount DOUBLE COMMENT 'Transaction amount',
        created_ts TIMESTAMP COMMENT 'Creation timestamp'
    )
    USING iceberg
    COMMENT 'Demo table for testing Iceberg operations'
""")

print(f"✓ Table created: {full_table}")

# Verify table exists
spark.sql(f"DESCRIBE EXTENDED {full_table}").show(truncate=False)

## 9. Create Iceberg Table - Advanced (MOR)

Create an Iceberg table with Merge-on-Read (MOR) configuration for optimized write performance.

In [ ]:
# Define advanced table details
advanced_table_name = "demo_table_mor"
full_advanced_table = f"{catalog_name}.{schema_name}.{advanced_table_name}"

print(f"Creating MOR table: {full_advanced_table}")

# Create table with MOR configuration
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {full_advanced_table} (
        commit_id INT COMMENT 'Commit identifier',
        row_id BIGINT COMMENT 'Row identifier',
        payload STRING COMMENT 'Data payload',
        ts TIMESTAMP COMMENT 'Timestamp'
    )
    USING iceberg
    TBLPROPERTIES (
        'format-version' = '2',
        'write.delete.mode' = 'merge-on-read',
        'write.update.mode' = 'merge-on-read',
        'write.merge.mode' = 'merge-on-read'
    )
    COMMENT 'Demo table with Merge-on-Read configuration'
""")

print(f"✓ MOR table created: {full_advanced_table}")

# Show table properties
print("\nTable properties:")
spark.sql(f"SHOW TBLPROPERTIES {full_advanced_table}").show(truncate=False)

## 10. Insert Sample Data

Insert test data into the created tables to verify functionality.

In [ ]:
from datetime import datetime

# Insert data into simple table
print(f"Inserting data into {full_table}...")

spark.sql(f"""
    INSERT INTO {full_table} VALUES
        (1, 'Alice', 100.50, TIMESTAMP '2026-04-11 10:00:00'),
        (2, 'Bob', 250.75, TIMESTAMP '2026-04-11 11:00:00'),
        (3, 'Charlie', 175.25, TIMESTAMP '2026-04-11 12:00:00'),
        (4, 'Diana', 300.00, TIMESTAMP '2026-04-11 13:00:00'),
        (5, 'Eve', 425.50, TIMESTAMP '2026-04-11 14:00:00')
""")

print("✓ Data inserted successfully")

# Query and display data
print(f"\nData in {full_table}:")
spark.sql(f"SELECT * FROM {full_table} ORDER BY id").show()

## 11. Verify Spark Configuration

Check that Iceberg Spark extensions are properly configured.

In [ ]:
# Check Spark SQL extensions
extensions = spark.conf.get("spark.sql.extensions", "NOT_SET")
print(f"Spark SQL Extensions: {extensions}")

# Verify Iceberg is available
if "IcebergSparkSessionExtensions" in extensions:
    print("✓ Iceberg Spark extensions are enabled")
else:
    print("⚠ Iceberg Spark extensions may not be properly configured")

## 12. Summary and Cleanup

Display summary of created objects and optionally clean up resources.

In [ ]:
print("="*80)
print("SUMMARY")
print("="*80)
print(f"\n✓ Catalog: {catalog_name}")
print(f"✓ Schema: {schema_name}")
print(f"✓ Tables created:")
print(f"  - {full_table}")
print(f"  - {full_advanced_table}")
print("\nAll operations completed successfully!")

# List all tables in the schema
print(f"\nTables in {catalog_name}.{schema_name}:")
spark.sql(f"SHOW TABLES IN {catalog_name}.{schema_name}").show(truncate=False)

## Optional: Cleanup

Uncomment and run the following cell to drop the created tables and schema.

In [ ]:
# # Uncomment to cleanup
# print("Cleaning up...")
# spark.sql(f"DROP TABLE IF EXISTS {full_table}")
# spark.sql(f"DROP TABLE IF EXISTS {full_advanced_table}")
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema_name} CASCADE")
# print("✓ Cleanup complete")